# 01. Snowflake: The Semantic View (Bidirectional Demo)

One semantic model, two platforms, changes flowing both ways. Nobody re-implements
anybody else's metric.

```
        SNOWFLAKE                    S3 (your bucket)                 DATABRICKS
  +-------------------+                                          +-------------------+
  |   SALES_SV        |                                          | sales_metric_view |
  |  (Semantic View)  |                                          |   (Metric View)   |
  +---------+---------+                                          +---------+---------+
            |                   ossie/sales_model.yaml                     |
            +--- export -----------> [ Ossie ] <----------- export --------+
            +--- import <----------- [  YAML ] ------------> import -------+
            |                                                             |
  +---------+---------+          iceberg/customers/                +------+---------+
  | CUSTOMERS, ORDERS |  ------> iceberg/orders/     <------------ | customers,     |
  | (Iceberg tables)  |          (Parquet + metadata)              | orders         |
  +-------------------+                                            +----------------+
                              same physical files, no copy
```

Two things are shared: the **data**, through Iceberg on S3, and the **meaning**, through
an Apache Ossie file on the same bucket.

You will visit three notebooks in this order:

| Order | Notebook | Platform | What happens |
|---|---|---|---|
| 1 | `01_snowflake_semantic_view` (this one) | Snowflake | Show the data and the semantic view |
| 2 | `02_databricks_metric_view` | Databricks | Same data, no metric view yet |
| 3 | back here | Snowflake | Export to Ossie |
| 4 | `02_databricks_metric_view` | Databricks | Build the metric view, add a measure, export |
| 5 | back here | Snowflake | Import, and the Databricks measure appears |
| 6 | `03_snowflake_automation` | Snowflake | Automate it, add another metric |
| 7 | `02_databricks_metric_view` | Databricks | The Snowflake metric arrived on its own |

Setup must already have been run: `00_snowflake_setup.sql`, then
`00_databricks_setup.ipynb`.

## Step 1 - Set your database and schema

In [ ]:
DATABASE           = "DEMOS"
SCHEMA             = "EXT_SEMANTIC_INTEROP"
SEMANTIC_VIEW_NAME = "SALES_SV"
STAGE_NAME         = "OSSIE_S3_STAGE"
MODEL_FILE         = "sales_model.yaml"

print(f"Working in {DATABASE}.{SCHEMA}")

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;
USE SCHEMA {{DATABASE}}.{{SCHEMA}};

## Step 2 - The data is Iceberg, on our own S3 bucket

Not Snowflake-internal tables. Snowflake writes Parquet and Iceberg metadata to a bucket
we own, and any Iceberg-capable engine can read it. Databricks is about to read these
exact files.

In [ ]:
%%sql -r dataframe_2
SHOW ICEBERG TABLES IN SCHEMA {{DATABASE}}.{{SCHEMA}};

In [ ]:
%%sql -r dataframe_3
-- Small enough to check by eye. 4 customers, 10 orders, integers only.
SELECT 
    * 
FROM 
    {{DATABASE}}.{{SCHEMA}}.CUSTOMERS 
ORDER BY customer_id;

In [ ]:
%%sql -r dataframe_4
SELECT c.region,
       SUM(o.order_amount) AS total_order_amount,
       COUNT(o.order_id)   AS order_count
  FROM {{DATABASE}}.{{SCHEMA}}.ORDERS o
  JOIN {{DATABASE}}.{{SCHEMA}}.CUSTOMERS c USING (customer_id)
 GROUP BY c.region ORDER BY c.region;
-- EAST 750/5, WEST 700/5 -- remember these two numbers

## Step 3 - The same two numbers, from the semantic view

`SALES_SV` holds the business definitions: `TOTAL_ORDER_AMOUNT` and `ORDER_COUNT`, the
region dimension, and the orders-to-customers relationship. Consumers ask for metrics by
name and never write the join or the aggregate themselves.

In [ ]:
%%sql -r dataframe_5
SELECT * FROM SEMANTIC_VIEW(
  {{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}}
  DIMENSIONS customers.region
  METRICS orders.total_order_amount, orders.order_count
) ORDER BY region;
-- same numbers, but now they come from a governed definition

> ### Switch to Databricks
>
> Open **`02_databricks_metric_view.ipynb`** and run its **Step 1**, to show that
> Databricks reads the same data and has no semantic model of its own.
>
> Then come back here.

## Step 4 - How the model itself crosses the boundary

The data is already shared. The definitions are not, yet.

- The interchange format is [Apache Ossie](https://github.com/apache/ossie), an open
  spec for semantic models.
- The file lives on the same S3 bucket as the Iceberg data: `ossie/sales_model.yaml`.
- **Snowflake reads and writes Ossie natively.** No connector, no ETL job, no external
  library. It is one function call in each direction.

Here is the entire export, as a single function: no arguments beyond the view name.

In [ ]:
SELECT SYSTEM$READ_OSSIE_YAML_FROM_SEMANTIC_VIEW(
  '{{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}}'
) AS ossie_yaml;
-- -- that is the whole semantic model, in an open format, from one function

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

# Get the YAML string from cell 9's result
yaml_text = dataframe_6.collect()[0][0] if not isinstance(dataframe_6, pd.DataFrame) else dataframe_6.iloc[0, 0]

# Display with YAML syntax highlighting
display(Markdown(f"```yaml\n{yaml_text}\n```"))

### Write it to S3

For this first export we do it by hand, so it is obvious there is no magic: the same
function, wrapped in a `COPY INTO` that lands the text on the stage as one file.

In [ ]:
%%sql -r dataframe_7
COPY INTO @{{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}}/{{MODEL_FILE}}
FROM (
  SELECT SYSTEM$READ_OSSIE_YAML_FROM_SEMANTIC_VIEW(
    '{{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}}'
  )
)
FILE_FORMAT = (TYPE = CSV FIELD_DELIMITER = NONE RECORD_DELIMITER = NONE
               ESCAPE_UNENCLOSED_FIELD = NONE COMPRESSION = NONE)
SINGLE = TRUE OVERWRITE = TRUE;

In [ ]:
%%sql -r dataframe_8
ALTER STAGE {{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}} REFRESH;

SELECT relative_path, size, last_modified
  FROM DIRECTORY(@{{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}})
 ORDER BY last_modified DESC;
-- sales_model.yaml is now on S3, next to the Iceberg data

### What happens next, on the other side

Databricks reads `sales_model.yaml` from this same bucket and builds a Unity Catalog
Metric View from it. The dimensions, the join and both measures come across; nobody
retypes a definition, and there is no pipeline moving rows.

The Metric View that appears over there is the same model you are looking at here.

> ### Switch to Databricks
>
> Open **`02_databricks_metric_view.ipynb`** and run its **Step 2**, which converts this
> file into a Metric View, adds a `TOTAL_QUANTITY` measure on the Databricks side, and
> exports the updated model back to S3.
>
> Then come back here.

## Step 5 - Import the Databricks change

Databricks added `TOTAL_QUANTITY` and wrote the model back to `sales_model.yaml`. Import
it, which is again a single native function.

`SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML` replaces `SALES_SV` in place, so the view
name and everything pointing at it stay valid.

In [ ]:
%%sql -r dataframe_9
-- Read the file the Databricks team just wrote.
SET yaml_content = (
  SELECT $1 FROM @{{DATABASE}}.{{SCHEMA}}.{{STAGE_NAME}}/{{MODEL_FILE}}
  (FILE_FORMAT => '{{DATABASE}}.{{SCHEMA}}.RAW_TEXT_FMT')
);

-- One function, and the semantic view is theirs plus ours.
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_OSSIE_YAML('{{DATABASE}}.{{SCHEMA}}', $yaml_content);

In [ ]:
%%sql -r dataframe_10
SHOW SEMANTIC METRICS IN {{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}};

SELECT $5 AS metric_name, $6 AS data_type
  FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));
-- three metrics now: TOTAL_QUANTITY came from Databricks

In [ ]:
%%sql -r dataframe_11
SELECT * FROM SEMANTIC_VIEW(
  {{DATABASE}}.{{SCHEMA}}.{{SEMANTIC_VIEW_NAME}}
  DIMENSIONS region
  METRICS total_quantity, total_order_amount, order_count
) ORDER BY region;
-- EAST 12/750/5, WEST 11/700/5
-- a measure authored in Databricks, queried through a Snowflake Semantic View

## Where we are

A metric defined in Snowflake reached Databricks, and a measure defined in Databricks
reached Snowflake. Both were single function calls on this side.

What we have not done is make it happen without a human running cells.

Open **`03_snowflake_automation.ipynb`** to schedule it.